<a href="https://colab.research.google.com/github/AnugrahaSaji/AnugrahaSaji/blob/main/DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qiskit qiskit-aer pycryptodome

In [ ]:
# ================================
# QUANTUM-ASSISTED E-VOTING DEMO
# ================================

# ---- IMPORTS ----
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
import hashlib
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

print("\n====== Quantum-Assisted E-Voting Prototype ======\n")

# ---- SIMULATED DATABASE (ONE VOTER) ----
database = {
    "VOTER1": {
        "name": "Demo User",
        "has_voted": False
    }
}

votes_storage = []

# ---- STEP 1: LOGIN ----
voter_id = input("Enter Voter ID: ")

if voter_id not in database:
    print("❌ Invalid Voter")
    raise SystemExit

if database[voter_id]["has_voted"]:
    print("❌ Voter has already voted")
    raise SystemExit

print("✅ Voter Verified\n")

# ---- STEP 2: QUANTUM KEY GENERATION ----
qc = QuantumCircuit(2,2)
qc.h(0)
qc.cx(0,1)
qc.measure([0,1],[0,1])

sim = Aer.get_backend('aer_simulator')
qc = transpile(qc, sim)

result = sim.run(qc, shots=1).result()
counts = result.get_counts()

quantum_key = list(counts.keys())[0]

print("🔐 Quantum Key Generated:", quantum_key)

# ---- STEP 3: HASH THE KEY (SHA-256) ----
hashed_key = hashlib.sha256(quantum_key.encode()).digest()
print("🔑 Hashed Key Created\n")

# ---- STEP 4: AES CHALLENGE AUTHENTICATION ----
challenge = get_random_bytes(16)

cipher = AES.new(hashed_key, AES.MODE_GCM)
ciphertext, tag = cipher.encrypt_and_digest(challenge)

# Simulated client decrypt
cipher2 = AES.new(hashed_key, AES.MODE_GCM, nonce=cipher.nonce)
response = cipher2.decrypt_and_verify(ciphertext, tag)

if response == challenge:
    print("✅ Authentication Successful\n")
else:
    print("❌ Authentication Failed")
    raise SystemExit

# ---- STEP 5: CAST VOTE ----
print("Candidates: A, B, C")
vote = input("Enter your vote: ")

# Encrypt Vote
cipher3 = AES.new(hashed_key, AES.MODE_GCM)
encrypted_vote, tag2 = cipher3.encrypt_and_digest(vote.encode())

votes_storage.append(encrypted_vote.hex())

# Mark voter as voted
database[voter_id]["has_voted"] = True

print("\n🗳 Vote Successfully Cast!")
print("🔒 Encrypted Vote Stored:", encrypted_vote.hex())

# ---- STEP 6: SESSION DESTROYED ----
quantum_key = None
hashed_key = None

print("\n🔚 Session Destroyed")
print("\n====== DEMO COMPLETE ======\n")


====== Quantum-Assisted E-Voting Prototype ======

Enter Voter ID: VOTER1
✅ Voter Verified

🔐 Quantum Key Generated: 00
🔑 Hashed Key Created

✅ Authentication Successful

Candidates: A, B, C
Enter your vote: A

🗳 Vote Successfully Cast!
🔒 Encrypted Vote Stored: 90

🔚 Session Destroyed

====== DEMO COMPLETE ======

